# Entrenamiento de modelos para extracción de datos de INE

Este notebook entrena **dos modelos** a partir de un dataset **100% sintético** (no usa ninguna INE real: eso sería un problema legal/de privacidad):

1. **Segmentación (UNet)** — localiza y recorta la credencial dentro de una foto. Reutiliza la misma arquitectura UNet que ya usa este repositorio para las credenciales de Albania/España (`id_card.ipynb`), adaptada a las INE.
2. **Detección de campos (Faster R-CNN)** — localiza la caja de cada campo individual (NOMBRE, CURP, CLAVE DE ELECTOR, etc.) dentro de la credencial ya recortada, para poder aplicar OCR campo por campo en vez de sobre todo el texto junto.

El dataset se genera con `ine_model/synthetic_data.py`: dibuja una plantilla esquemática de INE (sin holograma oficial, sin diseño de seguridad real), la rellena con datos falsos (Faker + generador de CURP/clave de elector con formato válido) y la compone sobre fondos aleatorios con perspectiva/rotación/ruido, igual que hace el dataset MIDV2020 que ya usa este repo.

**Recomendado: ejecutar con GPU** (Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU). El detector de campos en particular es lento en CPU.


## 0. Preparar el entorno

In [ ]:
import os

REPO_URL = "https://github.com/dizo/id-card-information-extraction.git"
REPO_DIR = "id-card-information-extraction"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}


In [ ]:
# Torch/TensorFlow ya vienen preinstalados en Colab; solo instalamos lo que falta.
!pip install -q -r ine_model/requirements.txt


In [ ]:
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt

from ine_model.synthetic_data import generate_dataset, FIELD_NAMES
from ine_model.segmentation_model import (
    train_segmentation_model,
    predict_mask,
    crop_card_from_mask,
)
from ine_model.detection_model import (
    train_detection_model,
    load_detection_model,
    predict_fields,
)

print("Campos a detectar:", FIELD_NAMES)


## 1. Generar el dataset sintético

Ajusta `N_SAMPLES`: unas cientos de imágenes alcanzan para probar que todo funciona; para resultados que generalicen bien a fotos reales se recomiendan **varios miles** (3000–8000). Con 4000 imágenes la generación toma unos ~10 minutos en CPU.

Si quieres guardar el dataset en Google Drive para no regenerarlo cada vez que abras el notebook, monta Drive primero y cambia `DATASET_DIR`.


In [ ]:
# from google.colab import drive
# drive.mount('/content/gdrive')
# DATASET_DIR = "/content/gdrive/MyDrive/ine_dataset"

N_SAMPLES = 4000  # baja esto (p.ej. 200) para una corrida rápida de prueba
DATASET_DIR = "ine_dataset"

t0 = time.time()
ann_path = generate_dataset(N_SAMPLES, DATASET_DIR, seed=42, out_size=(900, 700))
print(f"Generadas {N_SAMPLES} imágenes en {time.time() - t0:.1f}s -> {ann_path}")


### Vista rápida de una muestra (imagen + máscara + cajas de campos)

In [ ]:
import json

with open(ann_path, encoding="utf-8") as f:
    annotations = json.load(f)

sample = annotations[0]
img = cv2.imread(os.path.join(DATASET_DIR, "images", sample["image"]))
mask = cv2.imread(os.path.join(DATASET_DIR, "masks", sample["mask"]), cv2.IMREAD_GRAYSCALE)

vis = img.copy()
for box in sample["boxes"]:
    x1, y1, x2, y2 = box["bbox"]
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0].set_title("Imagen"); axes[0].axis("off")
axes[1].imshow(mask, cmap="gray"); axes[1].set_title("Máscara (segmentación)"); axes[1].axis("off")
axes[2].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); axes[2].set_title("Cajas de campos (detección)"); axes[2].axis("off")
plt.show()


## 2. Entrenar el modelo de segmentación (localizar/recortar la credencial)

Guarda el mejor checkpoint (según `val_loss`) en `unet_ine.h5`.


In [ ]:
SEG_EPOCHS = 30
SEG_INPUT_SIZE = 256

seg_model = train_segmentation_model(
    DATASET_DIR,
    epochs=SEG_EPOCHS,
    input_size=SEG_INPUT_SIZE,
    batch_size=16,
    checkpoint_path="unet_ine.h5",
)


### Probar el modelo de segmentación en una imagen

In [ ]:
test_img = cv2.imread(os.path.join(DATASET_DIR, "images", annotations[1]["image"]))
pred_mask = predict_mask(seg_model, test_img, input_size=SEG_INPUT_SIZE)
cropped_card = crop_card_from_mask(test_img, pred_mask)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)); axes[0].set_title("Foto original"); axes[0].axis("off")
axes[1].imshow(pred_mask, cmap="gray"); axes[1].set_title("Máscara predicha"); axes[1].axis("off")
axes[2].imshow(cv2.cvtColor(cropped_card, cv2.COLOR_BGR2RGB)); axes[2].set_title("Credencial recortada"); axes[2].axis("off")
plt.show()


## 3. Entrenar el detector de campos (Faster R-CNN)

Detecta la caja de cada campo (NOMBRE, DOMICILIO, CURP, CLAVE DE ELECTOR, ...) dentro de la credencial ya recortada. Esto es lo que permite hacer OCR campo por campo en vez de sobre todo el texto, mejorando mucho la precisión frente al enfoque de regex sobre el texto completo que usa `webapp/ine_extractor.py` por defecto.

Es más lento de entrenar que la segmentación — usa GPU si puedes. Guarda el mejor checkpoint en `field_detector.pth`.


In [ ]:
DET_EPOCHS = 15

det_model = train_detection_model(
    DATASET_DIR,
    epochs=DET_EPOCHS,
    batch_size=8,
    checkpoint_path="field_detector.pth",
)


### Probar el detector de campos en una imagen

In [ ]:
test_card = cropped_card  # usa la credencial ya recortada por el modelo de segmentación
fields = predict_fields(det_model, test_card, score_threshold=0.5)

vis = test_card.copy()
for label, (x1, y1, x2, y2) in fields.items():
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 200, 0), 2)
    cv2.putText(vis, label, (x1, max(0, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"{len(fields)}/{len(FIELD_NAMES)} campos detectados")
plt.show()


## 4. Exportar los pesos para la aplicación web

`webapp/ine_extractor.py` busca automáticamente estos dos archivos y, si existen, los usa en vez del recorte heurístico + regex:

- `webapp/model/unet_ine.h5`
- `webapp/model/field_detector.pth`

Si no existen, la app sigue funcionando igual que antes (con la heurística de OpenCV + regex sobre el texto completo).


In [ ]:
import shutil

os.makedirs("webapp/model", exist_ok=True)
shutil.copy("unet_ine.h5", "webapp/model/unet_ine.h5")
shutil.copy("field_detector.pth", "webapp/model/field_detector.pth")
print("Copiados a webapp/model/. Si corriste esto en Colab, descárgalos y súbelos a tu copia local del repo:")
print(" - webapp/model/unet_ine.h5")
print(" - webapp/model/field_detector.pth")

# En Colab, para descargarlos directamente:
# from google.colab import files
# files.download("webapp/model/unet_ine.h5")
# files.download("webapp/model/field_detector.pth")


## 5. (Opcional) Evaluar con tus propias fotos reales

Los datos sintéticos generan un modelo razonable, pero siempre hay una brecha (*domain gap*) frente a fotos reales. Si tienes fotos de **tu propia credencial** (nunca de la de alguien más, salvo que tengas su autorización explícita), puedes usarlas aquí como una prueba de sanidad cualitativa — no se necesita etiquetar nada, solo mirar si el recorte y las cajas de campos se ven razonables.

**Estas fotos NUNCA deben subirse a este repositorio.** Colócalas en una carpeta local `real_samples/` (ya está en `.gitignore`) y no las incluyas en ningún commit.


In [ ]:
import glob

REAL_SAMPLES_DIR = "real_samples"  # carpeta local, NO se sube al repo
real_files = sorted(
    glob.glob(os.path.join(REAL_SAMPLES_DIR, "*.jpg"))
    + glob.glob(os.path.join(REAL_SAMPLES_DIR, "*.jpeg"))
    + glob.glob(os.path.join(REAL_SAMPLES_DIR, "*.png"))
)
print(f"{len(real_files)} foto(s) encontradas en {REAL_SAMPLES_DIR}/")


In [ ]:
for path_ in real_files[:5]:  # limita cuántas se muestran a la vez
    real_img = cv2.imread(path_)
    if real_img is None:
        print("No se pudo leer:", path_)
        continue

    real_mask = predict_mask(seg_model, real_img, input_size=SEG_INPUT_SIZE)
    real_crop = crop_card_from_mask(real_img, real_mask)

    vis = real_crop.copy()
    fields_found = {}
    if det_model is not None:
        fields_found = predict_fields(det_model, real_crop, score_threshold=0.5)
        for label, (x1, y1, x2, y2) in fields_found.items():
            cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 200, 0), 2)
            cv2.putText(vis, label, (x1, max(0, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(cv2.cvtColor(real_img, cv2.COLOR_BGR2RGB)); axes[0].set_title(os.path.basename(path_)); axes[0].axis("off")
    axes[1].imshow(real_mask, cmap="gray"); axes[1].set_title("Máscara predicha"); axes[1].axis("off")
    axes[2].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); axes[2].set_title(f"{len(fields_found)} campos detectados"); axes[2].axis("off")
    plt.show()


Si el recorte o las cajas no calzan bien con tus fotos reales, revisa la sección **Siguientes pasos** más abajo: ajustar `FIELD_BOXES_FRAC`, generar más datos sintéticos, o continuar el entrenamiento (fine-tuning) partiendo de estos checkpoints con un pequeño set de fotos reales etiquetadas manualmente (por ejemplo con [VGG Image Annotator](https://www.robots.ox.ac.uk/~vgg/software/via/), la misma herramienta cuyo formato de anotación ya usa este repo en `create_masks.ipynb`).


## Siguientes pasos para mejorar la precisión

- **Aumenta `N_SAMPLES` y las épocas.** Más datos sintéticos y más entrenamiento generalizan mejor a fotos reales.
- **Ajusta el layout.** Si al probar con tu propia credencial (con tu autorización, nunca con la de alguien más) el recorte o las cajas de campos no calzan bien, edita `FIELD_BOXES_FRAC` en `ine_model/template.py` para que coincida mejor con el formato real y vuelve a generar el dataset.
- **Fine-tuning con fotos reales propias.** Puedes tomar unas cuantas fotos de tu propia credencial en distintos fondos/ángulos, etiquetar manualmente la máscara de la tarjeta y las cajas de los campos, y continuar el entrenamiento (`model.fit` / el loop de `train_detection_model`) partiendo de estos checkpoints para cerrar el "domain gap" entre datos sintéticos y fotos reales.
- **Aumenta la resolución** (`SEG_INPUT_SIZE`, `out_size` del dataset) si los campos de texto quedan muy pequeños para el OCR.
